# Module 15: Heterogeneous Effects, and the Temptation to Find Them

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Given four estimates a reader will rank them and ask why the program worked
best somewhere. **Here the answer is that it did not**, and the module is
about how to establish that rather than assert it.

The same machinery then shows how easily a subgroup analysis manufactures a
finding.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Four estimates of one number

In [ ]:
import scipy.stats as st

for a in KEEP:
    d[f"s_{a}"] = ((d["agency_id"] == a) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "phase")).astype(float)
d["settled"] = ((d["agency_id"].isin(KEEP)) & (d["period"] == "after")).astype(float)

terms = " + ".join(f"s_{a}" for a in KEEP)
zh = smf.glm("n_uof ~ C(agency_id)+C(year_month)+phase+" + terms, d,
             family=sm.families.Poisson(), offset=d["lo"]).fit()
z0 = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase", d,
             family=sm.families.Poisson(), offset=d["lo"]).fit()

rows = []
for a in KEEP:
    lo, hi = zh.conf_int().loc[f"s_{a}"]
    rows.append({"agency": NAME[a].split()[0],
                 "estimate": f"{pct(zh.params[f's_{a}']):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
lo, hi = z0.conf_int().loc["settled"]
rows.append({"agency": "POOLED", "estimate": f"{pct(z0.params['settled']):+.1f}%",
             "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
print(f"  the planted effect is {TRUTH:+.1f} percent at every one of them\n")
pd.DataFrame(rows).set_index("agency")

Twelve points of spread, and the true effect is identical at all four.

## 3. The test, rather than the ranking

In [ ]:
lr = 2 * (zh.llf - z0.llf)
pv = 1 - st.chi2.cdf(lr, len(KEEP) - 1)
print(f"  likelihood ratio test, one common effect against four separate ones")
print(f"    chi squared {lr:.2f} on {len(KEEP) - 1} degrees of freedom, p = {pv:.3f}")
print(f"\n  the four estimates span "
      f"{max(pct(zh.params[f's_{a}']) for a in KEEP) - min(pct(zh.params[f's_{a}']) for a in KEEP):.1f} points")

**No evidence of heterogeneity**, and the spread of twelve points is what four
noisy estimates of one number look like.

The test is the sentence to report. "The effect ranged from 9 to 21 percent
across agencies" is true, misleading, and the kind of thing that ends up in a
summary.

## 4. What the test could have detected

As always, a passing test is worth nothing until its power is known.

In [ ]:
rng = np.random.default_rng(6)


# Resample from the fitted common effect model, so the simulated worlds are
# homogeneous by construction rather than inheriting this sample's accidents.
BASE = z0.fittedvalues.values


def detect(spread_pct, reps=120):
    """Plant a spread of effects across the four agencies and see how often
    the homogeneity test rejects."""
    effs = np.linspace(0.12 - spread_pct / 200, 0.12 + spread_pct / 200, len(KEEP))
    hits = 0
    for _ in range(reps):
        s = d.copy()
        mult = np.ones(len(s))
        for a, e in zip(KEEP, effs):
            sel = ((s["agency_id"] == a) & (s["period"] == "after")).values
            mult[sel] = (1 - e) / 0.88
        s["y"] = rng.poisson(np.maximum(BASE * mult, 0.01))
        zh2 = smf.glm("n_uof ~ C(agency_id)+C(year_month)+phase+" + terms,
                      s.assign(n_uof=s["y"]), family=sm.families.Poisson(),
                      offset=s["lo"]).fit()
        z02 = smf.glm("n_uof ~ C(agency_id)+C(year_month)+settled+phase",
                      s.assign(n_uof=s["y"]), family=sm.families.Poisson(),
                      offset=s["lo"]).fit()
        hits += (1 - st.chi2.cdf(2 * (zh2.llf - z02.llf), len(KEEP) - 1)) < 0.05
    return 100 * hits / reps


for spread in [0, 10, 20, 30]:
    print(f"  a planted spread of {spread:2d} points is detected "
          f"{detect(spread):3.0f} percent of the time")

A ten point spread is found one time in ten. A twenty point spread is a coin
flip. Only a thirty point spread is found reliably.

**So "no evidence of heterogeneity" here means the effect does not differ
across agencies by something like thirty points**, which is a much weaker
statement than "the effect is the same everywhere" and is the one the data
supports.

## 5. Subgroups, and how easily one appears

In [ ]:
prof = profile.set_index("agency_id")
splits = {"large agencies": lambda a: prof.loc[a, "sworn_officers"] > 100,
          "western region": lambda a: prof.loc[a, "region"] == "West",
          "municipal police": lambda a: prof.loc[a, "agency_type"] == "Municipal Police",
          "above median violent crime":
              lambda a: prof.loc[a, "violent_crime_rate_per_1000"]
                        > prof["violent_crime_rate_per_1000"].median(),
          "above median population":
              lambda a: prof.loc[a, "population_served"]
                        > prof["population_served"].median()}
rows = []
for lab, rule in splits.items():
    ing = [a for a in KEEP if rule(a)]
    outg = [a for a in KEEP if not rule(a)]
    if not ing or not outg:
        continue
    ei = fit(d[d["agency_id"].isin(ing + COMPARISON)], ing)[0]
    eo = fit(d[d["agency_id"].isin(outg + COMPARISON)], outg)[0]
    rows.append({"split": lab, "in the subgroup": f"{ei:+.1f}%",
                 "not in it": f"{eo:+.1f}%", "difference": round(ei - eo, 1)})
print(f"  the true effect is {TRUTH:+.1f} percent in every subgroup\n")
pd.DataFrame(rows).set_index("split")

Differences of three to eight points appear across splits where the truth is
identical, and the largest, municipal police at 12.2 against 20.6 percent, is
the kind of gap that gets a sentence in a summary.

**With four treated units, any binary split produces groups small enough to
differ by several points**, and a search across five splits finds one that
looks substantial. None of these is a finding.

| Protection | Cost |
|---|---|
| Pre register the subgroups | none, and it is the only real protection |
| Report every split examined | none |
| Correct for multiplicity | the surviving claims are weaker, correctly |
| Require a mechanism in advance | some genuine discoveries are missed |
| Report the homogeneity test first | none |

## Exercise

Estimate how often a search across five splits turns up a difference of at
least ten points when the effect is genuinely constant.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rng2 = np.random.default_rng(14)
    pairs = [(["A001", "A002"], ["A004", "A010"]),
             (["A001", "A004"], ["A002", "A010"]),
             (["A001", "A010"], ["A002", "A004"])]
    hits = 0
    reps = 60
    for _ in range(reps):
        s = d.copy()
        s["y"] = rng2.poisson(np.maximum(BASE, 0.01))
        found = False
        for ing, outg in pairs:
            ei = fit(s[s["agency_id"].isin(ing + COMPARISON)], ing, outcome="y")[0]
            eo = fit(s[s["agency_id"].isin(outg + COMPARISON)], outg, outcome="y")[0]
            if abs(ei - eo) >= 8:
                found = True
        hits += found
    print(f"  the effect is constant across all four agencies by construction\n")
    print(f"  a search across only three splits finds a gap of 8 points or more")
    print(f"  in {100 * hits / reps:.0f} percent of simulated datasets")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

A search across only three splits turns up an eight point gap in a large
share of simulated datasets, and the effect is constant in every one of them.

**That is the base rate a subgroup finding has to beat**, and it is rarely
computed. A report that says "the effect was larger at municipal agencies,
20.6 percent against 12.2" is describing something that happens routinely when
nothing is there.

The simulation is four lines and it converts an argument into a number. When
a subgroup difference is claimed, ask what the search space was and what gap
that search produces under a constant effect. If the observed gap is inside
that range, there is nothing to explain.

</details>

---

**Next:** [Module 16: The Causal Claim, What You Can Defend](Module_16_The_Causal_Claim.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*